# **Introduction to Agentic Workflows**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [What is Agentic AI?](#What-is-Agentic-AI?)
- [Degrees of Autonomy](#Degrees-of-Autonomy)
- [Key Benefits of Agentic AI](#Key-Benefits-of-Agentic-AI)
  - [Much Better Performance](#Much-Better-Performance)
  - [Faster than Humans because of Parallelization](#Faster-than-Humans-because-of-Parallelization)
  - [Modular: Can Add/Update Tools, Swap out Models](#Modular:-Can-Add/Update-Tools,-Swap-out-Models)
- [Applications of Agentic AI](#Applications-of-Agentic-AI)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# What is Agentic AI?

What is agentic AI and why are agentic AI workflows so powerful? Let's take a look. The way that many of us use large language models today is by prompting it to say, `"Please type out an essay on topic X from start to finish in one go, without using backspace"`. It turns out that we as people, we don't do our best writing like that by being forced to write in this completely linear order and nor do AI models. But despite the difficulty of being constrained to write in this way, our LLMs do surprisingly well.

In contrast, with an agentic workflow, the process might look like below:
* You may ask it to first write an essay outline on a certain topic
* Then ask if it needs to do any web research
* After doing some web research and maybe downloading some web pages, ask it to write the first draft
* Then to read the first draft and see what parts need revision or more research
* Then revise the draft and so on.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-01.png" width="800">

This type of workflow is more akin to doing some thinking and some research and then doing some revision and then doing some more thinking and so on. And with this iterative process, it turns out that an agentic workflow can take longer, but it delivers a much better work product. So an agentic AI workflow is a process where an LLM based app executes multiple steps to complete a task.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-02.png" width="800">

One of the key skills you learn in this course is how to take a complex task like writing an essay and breaking it down into smaller steps for agentic workflows to execute one step at a time to then get the work output that you want. And knowing how to decompose the task into steps and how to build the components to execute the individual steps well turns out to be a tricky but important skill that will determine your ability to build agentic workflows for a huge range of exciting applications.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Degrees of Autonomy

Agents can be autonomous to different degrees. A few years ago, there was a growing controversial debate within the AI community about what is an agent, and what is a true agent. This debate was unnecessary. When we use the term **agentic**, i.e. if we use it as an adjective rather than a binary (it's either an agent or not), then we're going to have to acknowledge that systems can be agentic to different degrees. Instead of debating on what is a true agent, we should just focus on actually building them.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-03.png" width="800">

Some agents can be less autonomous. So take the example of writing an essay about black holes. You can have a relatively simple agent to come up with a few web search terms or web search queries. Then you can hard code that
1. you call a web search engine
2. fetch some web pages
3. then use that to write an essay.

And this is an example of a less autonomous agent with a fully deterministic sequence of steps.

Then an agent can be more autonomous, where, given a request to write an essay about black holes,
1. you let the LLM decide, does it want to do a web search, or does it want to search recent news sources, or does it want to search for recent research papers? Based on that, not the human engineer, but the LLM chooses, in this case, to call a web search engine.
2. then after that, you may let the LLM decide how many web pages does it want to fetch, or if it fetches the PDF, does it need to call a tool, to convert the PDF to text? In this case, it fetches its 5 top web pages.
3. then it can write an essay
4. decide whether to reflect and improve, and maybe even go back to fetch more web pages,
5. then to finally produce an output.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-04.png" width="800">

And so even for this example of a research agent, we can see that some agents can be less autonomous, with a linear sequence of steps to be executed, determined by a programmer. Some can be more autonomous, where you trust the LLM to make more decisions, and the exact sequence of steps that happens may be even determined by the LLM, rather than in advance by the programmer. So for less autonomous systems, you will usually have all the steps predetermined in advance, and any functions it calls (tool use), like web search, might be hard-coded by the human engineer, and most of the autonomy is in what text the LLM generates.

At the end of the spectrum would be highly autonomous agents, where the agent makes many decisions autonomously, including, for example, deciding what is the sequence of steps it will carry out in order to write the essay. And there are some highly autonomous agents that can even write new functions, or sometimes create new tools that it can then execute. And somewhere in between are semi-autonomous agents, where it can make some decisions, choose tools, but the tools are usually more predefined.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-05.png" width="800">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Key Benefits of Agentic AI

## Much Better Performance

Data is collected from a coding benchmark called HumanEval that tests the ability of different LLMs to write code to carry out certain tasks. GPT 3.5 (model that the first publicly available version of ChaTGPT was based on) got 40% score on this benchmark with a non-agentic workflow. GPT-4, which is a much better model, leaps to 67% in this benchmark, also with a non-agentic workflow.

But it turns out that as large as the improvement was from GPT 3.5 to GPT-4, that improvement is dwarfed by what you can achieve by wrapping GPT 3.5 within an agentic workflow. Using different agentic techniques, you can prompt GPT 3.5 to write code and then maybe reflect on the code and figure out if you can improve it. Using techniques like that, you can actually get GPT 3.5 to get much higher levels of performance. Similarly, GPT-4 used in the context of an agentic workflow also does much better.

So even with today's best LLMs, an agentic workflow lets you get much better performance. And in fact, what we saw in this example was the improvement from one generation of model to another, which is huge, is still not as big a difference as implementing an agentic workflow on the previous generation of model.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-06.png" width="800">

## Faster than Humans because of Parallelization

Another benefit of using agentic workflows is that they can parallelize some tasks and thus do certain things much faster than a human. For example, if you ask an agentic workflow to write an essay about black holes, you might be able to have three LLMs run in parallel to generate ideas for web search terms to type into the search engine. We can fetch top 3 results from each of those three searches and use 9 LLMs to download those 9 webpages. It turns out that whereas a human doing this research would have to read these nine web pages sequentially or one at a time, when you're using an agentic workflow, you can actually parallelize all nine web page downloads and then finally feed all these things into an LLM to write an essay. So even though agentic workflows do take longer than truly non-agentic workflows or by direct generation by just prompting a single time, if you were to compare this type of agentic workflow to how a human would have to go about the task, the ability to parallelize tasks can actually let it do certain tasks much faster than the non-parallel sequential way that a single human might process this data.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-07.png" width="800">

## Modular: Can Add/Update Tools, Swap out Models

To build on this example, it turns out one of the things we can do when building agentic workflows is look at the individual components like the LLM and add, update or swap out components. So for example, when building agentic workflows, we can use multiple web search engines including Google, Bing, DuckDuckGo, Tavily, You.com etc. Or maybe instead of just doing three web searches, we can swap in a new News search engine so we can find out what's the latest news on recent breakthroughs on black hole science. And lastly, instead of using the same LLM for all of the different steps, we can try out different large language models and different LLM providers to see which one gives the best result for different steps of this system. Thus, the modular design of many agentic workflows also lets us add or update tools and sometimes swap out models.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-08.png" width="800">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Applications of Agentic AI

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-09.png" width="800">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Simple Agentic AI for Invoice Processing</em>
    </td>
  </tr>
</table>

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-10.png" width="800">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Handle Customer Order Inquiries with Clear Workflows</em>
    </td>
  </tr>
</table>

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-11.png" width="800">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Tackle Complex, Open-Ended Customer Service Tasks</em>
    </td>
  </tr>
</table>

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-12.gif" width="800">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Watch an Agent Use a Web Browser to Complete a Task</em>
    </td>
  </tr>
</table>

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-13.png" width="800">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[DeepLearning.AI: Agentic AI Course](https://www.deeplearning.ai/courses/agentic-ai)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/Agentic-AI-with-Andrew-Ng/data/images/01_Intro_to_Agentic_Workflows-05.png" width="800">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)